# Project: Carbon Footprint Prediction (India)
## Notebook 02: Feature Engineering and Target Alignment

[![Author](https://img.shields.io/badge/Author-Prakash%20Ukhalkar-blue.svg)](https://github.com/prakash-ukhalkar) [![GitHub Repository](https://img.shields.io/badge/GitHub-Repo-lightgrey)](https://github.com/prakash-ukhalkar/India-Agri-Carbon-Prediction) [![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://www.python.org/) [![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![NumPy](https://img.shields.io/badge/NumPy-Latest-lightblue)](https://numpy.org/) [![statsmodels](https://img.shields.io/badge/statsmodels-Latest-orange)](https://www.statsmodels.org/) [![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

---

**Objective:** Transform consolidated masters into a stationary, feature-rich dataset for ML modeling.

**Introduction:** This notebook focuses on the Feature Engineering phase. We will align the three thematic master files, handle the high dimensionality of the energy features, and engineer "Lag" and "Rolling Average" variables to capture the temporal momentum of carbon emissions. We will specifically focus on the Water-Energy-Food Nexus to create high-impact research features.

### Step 1: Environment Setup & Data Loading
Explain the importance of feature scaling and stationarity in time-series research.
In econometric and ML time-series predictions, ensuring stationarity (constant mean/variance) prevents spurious correlations. Feature engineering (lags, momentum) allows the model to learn from historical trends, which is essential for capturing "policy lag" in agricultural transitions.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DATA_DIR = '../data/processed/'

# Load raw masters
df_emi = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_emissions_panel.csv'))
df_ene = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_energy_activity.csv'))
df_res = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'master_land_water_resource.csv'))

print("Data Loaded successfully.")
print(f"Emissions: {df_emi.shape}, Energy: {df_ene.shape}, Resources: {df_res.shape}")


Data Loaded successfully.
Emissions: (3660, 20), Energy: (65, 321), Resources: (65, 30)


### Step 2: Target Variable Definition ($CO_2e$)
Define the primary dependent variable by aggregating specific "Element" codes from the Emissions Master (e.g., Emissions (CO2e) from energy and crops).

In [2]:
# Filter emission data for total CO2 equivalent indicators
# FIX (scope): the original filter only captured agri-attributed ENERGY emissions
# (Item = Electricity/Natural gas/Petroleum products). It excluded the biological/soil
# sources that dominate a country's agricultural GHG inventory: rice cultivation CH4,
# crop-residue burning CH4/N2O, and soil N2O from crop residues. We now include those
# so "Target_Total_CO2e" actually represents agricultural carbon footprint, not just
# on-farm energy use. Note: 'Crop residues (Emissions N2O)' is already the sum of
# 'Crop residues (Direct/Indirect emissions N2O)', so only the combined element is used
# to avoid double-counting.
target_elements = [
    'Emissions (CO2)', 'Emissions (CH4)', 'Emissions (N2O)',           # agri-energy (electricity, gas, petroleum)
    'Rice cultivation (Emissions CH4)',                                 # paddy methane
    'Burning crop residues (Emissions CH4)',                            # residue burning
    'Burning crop residues (Emissions N2O)',                            # residue burning
    'Crop residues (Emissions N2O)',                                    # soil N2O (direct+indirect, pre-summed)
]
df_target = df_emi[df_emi['Element'].isin(target_elements)].copy()

# FAOSTAT reports CH4/N2O in raw gas kt, NOT CO2-equivalent.
# Convert to CO2e using IPCC AR5 100-year GWP factors before summing,
# otherwise summing raw kt mixes physically different quantities.
GWP_AR5 = {
    'Emissions (CO2)': 1,
    'Emissions (CH4)': 28,
    'Emissions (N2O)': 265,
    'Rice cultivation (Emissions CH4)': 28,
    'Burning crop residues (Emissions CH4)': 28,
    'Burning crop residues (Emissions N2O)': 265,
    'Crop residues (Emissions N2O)': 265,
}
df_target['Value_CO2e'] = df_target['Value'] * df_target['Element'].map(GWP_AR5)

# Pivot to get a single total CO2e per year
df_grouped = df_target.groupby(['Year_Standardized'])['Value_CO2e'].sum().reset_index()
df_grouped.rename(columns={'Value_CO2e': 'Target_Total_CO2e'}, inplace=True)
df_grouped.set_index('Year_Standardized', inplace=True)

# FIX (scope consistency): FAOSTAT's agri-energy emissions series (CO2/CH4/N2O from
# Electricity/Natural gas/Petroleum products) only starts in 1990. Years before 1990
# would silently be missing that component, making the target inconsistent in scope
# across time (a ~45,000 kt level-shift artifact at the 1989->1990 boundary, not real
# emissions growth). Restrict to 1990+, where every component is consistently reported.
#
# FIX (no forecasted data in train/test): the electricity source (master_energy_activity.csv)
# flags 2023-2024 as `Is_Forecasted=True` (CAGR-extrapolated, not observed). Using those years
# as if they were real observations would let the model "predict" a value that is itself a
# straight-line extrapolation of history, inflating apparent skill. Cap the analysis at 2022,
# the last year every input series is a genuine observation.
LAST_RELIABLE_YEAR = 2022
df_grouped = df_grouped[(df_grouped.index >= 1990) & (df_grouped.index <= LAST_RELIABLE_YEAR)]

print("Target variable defined (GWP-converted, full agri-GHG scope, 1990-2022, forecasted years excluded). Shape:", df_grouped.shape)
df_grouped.head()


Target variable defined (GWP-converted, full agri-GHG scope, 1990-2022, forecasted years excluded). Shape: (33, 1)


,Target_Total_CO2e
Year_Standardized,
1990,189113.6686
1991,196976.7100
1992,200788.6662
1993,212024.9774
1994,218527.6413


### Step 3: Dimensionality Reduction (Energy & Activity)
Address the high dimensionality issue by removing features with >30% missing values or zero variance.

In [3]:
# Set Year as index
df_ene.set_index('Year_Standardized', inplace=True)
df_res.set_index('Year_Standardized', inplace=True)

# Feature preservation: slice to target range (1990-2023) before dimensional reduction
# This prevents dropping columns that are only available post-1990 (e.g. machinery, electricity)
target_years = df_grouped.index
df_ene_sliced = df_ene.loc[df_ene.index.isin(target_years)]
df_res_sliced = df_res.loc[df_res.index.isin(target_years)]

def reduce_dimensions(df, missing_thresh=0.3):
    # Drop completely empty columns
    df = df.dropna(axis=1, how='all')
    
    # Drop columns with >30% missing values in the sliced period
    limit = len(df) * (1 - missing_thresh)
    df = df.dropna(axis=1, thresh=limit)
    
    # Drop zero variance columns (constants)
    numeric_df = df.select_dtypes(include=[np.number])
    var = numeric_df.var()
    zero_var_cols = var[var == 0].index
    df = df.drop(columns=zero_var_cols, errors='ignore')
    
    return df

print(f"Energy before: {df_ene.shape} | Sliced: {df_ene_sliced.shape}")
df_ene_reduced = reduce_dimensions(df_ene_sliced)
print(f"Energy after reduction: {df_ene_reduced.shape}")

print(f"Resources before: {df_res.shape} | Sliced: {df_res_sliced.shape}")
df_res_reduced = reduce_dimensions(df_res_sliced)
print(f"Resources after reduction: {df_res_reduced.shape}")

Energy before: (65, 320) | Sliced: (33, 320)
Energy after reduction: (33, 48)
Resources before: (65, 29) | Sliced: (33, 29)
Resources after reduction: (33, 24)


### Step 4: Engineering "Intensity" & "Nexus" Features
Rationale for calculating Carbon Intensity (CO2e / Total Irrigated Area) and Energy Intensity (GWh / Hectare). These metrics standardize absolute growth allowing for fair academic "Impact Analysis." 

In [4]:
# Master merge for engineering
master_df = df_grouped.copy()
for d in [df_ene_reduced, df_res_reduced]:
    overlap = [c for c in d.columns if c in master_df.columns]
    master_df = master_df.join(d.drop(columns=overlap), how='inner')

import re
def clean_column_name(c):
    # Standardize superscripts, replacement chars, and % sign
    c = c.replace('\u00b3', '3').replace('\u00b2', '2').replace('\ufffd', '3').replace('%', 'pct')
    c = re.sub(r'[^a-zA-Z0-9_]', '_', c)
    c = re.sub(r'_+', '_', c)
    c = c.strip('_')
    return c

# Clean up column names to avoid special character issues in models
clean_cols = {c: clean_column_name(c) for c in master_df.columns}
master_df.rename(columns=clean_cols, inplace=True)

# Explicitly create intensity and nexus features using correct numeric columns
if 'Agricultural_land_ha_final' in master_df.columns:
    master_df['Emission_Intensity_Land'] = master_df['Target_Total_CO2e'] / (master_df['Agricultural_land_ha_final'] + 1e-5)
    print("Created Emission_Intensity_Land using Agricultural_land_ha_final")

if 'Agri_Electricity_Use_GWh' in master_df.columns and 'Estimated_agriculture_withdrawals_billion_m3' in master_df.columns:
    master_df['Energy_Intensity_Water'] = master_df['Agri_Electricity_Use_GWh'] / (master_df['Estimated_agriculture_withdrawals_billion_m3'] + 1e-5)
    print("Created Energy_Intensity_Water using Agri_Electricity_Use_GWh and Estimated_agriculture_withdrawals_billion_m3")

print("Master DF shape after adding nexus features:", master_df.shape)

Created Emission_Intensity_Land using Agricultural_land_ha_final
Created Energy_Intensity_Water using Agri_Electricity_Use_GWh and Estimated_agriculture_withdrawals_billion_m3
Master DF shape after adding nexus features: (33, 58)


### Step 5: Temporal Feature Engineering (Lags & Momentum)
Discuss why $t-1$ and $t-2$ lags are necessary to capture the "policy lag" in agricultural transitions. We create 1-year and 2-year lags for the primary emission and energy features, and calculate 3-year rolling means to smooth out climate-induced volatility.

In [5]:
# Select numeric columns
numeric_cols = list(master_df.select_dtypes(include=[np.number]).columns)
master_df = master_df[numeric_cols]

# Linear interpolation to fill any gaps smoothly
renewable_cols = [c for c in master_df.columns if 'Renewable' in c or 'renew' in c.lower()]
for col in renewable_cols:
    first_valid_index = master_df[col].first_valid_index()
    if first_valid_index is not None:
        master_df.loc[:first_valid_index, col] = master_df.loc[:first_valid_index, col].fillna(0)
    else:
        master_df[col] = master_df[col].fillna(0)

master_df = master_df.interpolate(method='linear', limit_direction='both')
print("Numeric dataset interpolated. Shape:", master_df.shape)

Numeric dataset interpolated. Shape: (33, 36)


### Step 6: Stationarity Testing (ADF Test)
Perform the Augmented Dickey-Fuller (ADF) test to check if the time series is stationary (a requirement for many ML algorithms).

In [6]:
from statsmodels.tsa.stattools import adfuller

TARGET_COL = 'Target_Total_CO2e'

# Define the training years partition to run ADF tests to prevent leakage from the test set
TEST_YEARS_HOLD_OUT = 5
train_years_raw = master_df.index[:-TEST_YEARS_HOLD_OUT]
df_train = master_df.loc[train_years_raw]

def test_stationarity(series, title):
    try:
        res = adfuller(series.dropna(), autolag='AIC')
        p_val = res[1]
        return p_val <= 0.05, p_val
    except Exception:
        return False, 1.0

# 1. Test Target variable on TRAINING data only
target_series = df_train[TARGET_COL]
target_stationary, target_p = test_stationarity(target_series, 'Target')
print(f"Target variable {TARGET_COL} level ADF p-value: {target_p:.4f} (Stationary: {target_stationary})")

master_stationary = master_df.copy()
master_stationary[f'{TARGET_COL}_diff'] = master_stationary[TARGET_COL].diff()

# 2. Programmatic Coordinated Differencing of Features
features_to_difference = []
stationary_features = []

for col in master_df.columns:
    if col == TARGET_COL:
        continue
    is_stationary, p_val = test_stationarity(df_train[col], col)
    if is_stationary:
        stationary_features.append(col)
    else:
        features_to_difference.append(col)

print(f"\nStationary features (levels kept): {len(stationary_features)}")
print(f"Non-stationary features (differenced): {len(features_to_difference)}")

# Apply differencing to non-stationary features
for col in features_to_difference:
    master_stationary[f'{col}_diff'] = master_stationary[col].diff()

# Keep only the stationary versions of variables
cols_to_keep = stationary_features + [f'{col}_diff' for col in features_to_difference] + [f'{TARGET_COL}_diff', TARGET_COL]
master_stationary = master_stationary[cols_to_keep]

# FIX #5: Drop the first row (NaN from differencing) INSTEAD of interpolating it.
# This prevents fabricating a synthetic first observation from future values.
rows_before = len(master_stationary)
master_stationary = master_stationary.iloc[1:]  # Drop first row (all diffs are NaN)
print(f"\nDropped first row after differencing: {rows_before} -> {len(master_stationary)} rows")

# Interpolate only genuine interior gaps (if any exist from source data misalignment)
interior_nans = master_stationary.isnull().sum().sum()
if interior_nans > 0:
    master_stationary = master_stationary.interpolate(method='linear')
    print(f"Interpolated {interior_nans} interior NaN values")
else:
    print("No interior NaN values — no interpolation needed")

print(f"Stationary dataset shape: {master_stationary.shape}")


Target variable Target_Total_CO2e level ADF p-value: 0.9326 (Stationary: False)



Stationary features (levels kept): 4
Non-stationary features (differenced): 31



Dropped first row after differencing: 33 -> 32 rows
No interior NaN values — no interpolation needed
Stationary dataset shape: (32, 37)


### Step 7: Feature Selection & Final Master Research Dataset (MRD)
To address the high-dimensional, low-sample-count problem (n ≪ p), we apply a
**two-stage feature selection** process after lag/rolling feature creation:

1. **Correlation Pruning**: For every feature pair with |r| > 0.90, drop the member
   less correlated with the target variable. This removes redundant scale-variants
   (e.g., kWh vs GWh, m³ vs billion m³) and near-duplicate lag/rolling features.

2. **VIF-Based Removal**: Iteratively remove the feature with the highest Variance
   Inflation Factor (VIF > 10) until all remaining features have VIF < 10. This
   ensures no severe multicollinearity in the final feature matrix.

**Target**: ≤ 20 features for ~26 training samples (n/p ≥ 1.3).


In [7]:
# 1. Exclude deterministic trend variables and target leakage
exclude_patterns = ['Year', 'Year_Code', 'StartYear', 'Crop_Year', 'Emission_Intensity_Land']
feature_cols = [c for c in master_stationary.columns if c not in [f'{TARGET_COL}_diff', TARGET_COL] and not any(p in c for p in exclude_patterns)]

metadata_cols = [f'{TARGET_COL}_diff', TARGET_COL, 'Emission_Intensity_Land', 'Emission_Intensity_Land_diff']
metadata_cols = [c for c in metadata_cols if c in master_stationary.columns]
master_final = master_stationary[metadata_cols + feature_cols].copy()

# 2. Construct Lags (1, 2) and Roll (3) for feature columns
for col in feature_cols:
    master_final[f'{col}_lag1'] = master_final[col].shift(1)
    master_final[f'{col}_lag2'] = master_final[col].shift(2)
    master_final[f'{col}_roll3_mean'] = master_final[col].rolling(window=3).mean()

# 3. Construct Lags and Roll for Target (autoregressive features)
master_final[f'{TARGET_COL}_diff_lag1'] = master_final[f'{TARGET_COL}_diff'].shift(1)
master_final[f'{TARGET_COL}_diff_lag2'] = master_final[f'{TARGET_COL}_diff'].shift(2)
master_final[f'{TARGET_COL}_diff_roll3_mean'] = master_final[f'{TARGET_COL}_diff'].shift(1).rolling(window=3).mean()

# 4. Drop rows with NaNs introduced by shifts (proper handling — no fabrication)
master_final.dropna(inplace=True)
print(f"After lag creation and dropna: {master_final.shape}")

# ═══════════════════════════════════════════════════════════════════
#  FIX #1: FEATURE SELECTION (Correlation Pruning + VIF)
# ═══════════════════════════════════════════════════════════════════
from statsmodels.stats.outliers_influence import variance_inflation_factor

active_target = f'{TARGET_COL}_diff'
# Identify feature columns (exclude target and metadata)
meta_cols = [c for c in master_final.columns if 'Target_Total_CO2e' in c or 'Emission_Intensity' in c]
feat_cols = [c for c in master_final.columns if c not in meta_cols]
feat_cols = [c for c in feat_cols if master_final[c].dtype in ['float64', 'int64', 'float32']]

print(f"\n--- Feature Selection ---")
print(f"Starting features: {len(feat_cols)}")

# Stage 1: Correlation Pruning (|r| > 0.90 between features)
corr_matrix = master_final[feat_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# For each pair with |r| > 0.90, drop the one with LOWER correlation to target
target_corr = master_final[feat_cols].corrwith(master_final[active_target]).abs()
cols_to_drop_corr = set()
for col in upper_tri.columns:
    high_corr_partners = upper_tri.index[upper_tri[col] > 0.90].tolist()
    for partner in high_corr_partners:
        if col in cols_to_drop_corr or partner in cols_to_drop_corr:
            continue
        # Drop the one less correlated with target
        if target_corr.get(col, 0) >= target_corr.get(partner, 0):
            cols_to_drop_corr.add(partner)
        else:
            cols_to_drop_corr.add(col)

feat_cols_pruned = [c for c in feat_cols if c not in cols_to_drop_corr]
print(f"After correlation pruning (|r|>0.90): {len(feat_cols_pruned)} features ({len(cols_to_drop_corr)} dropped)")

# Stage 2: VIF-based multicollinearity removal (iterative, VIF < 10)
MAX_VIF = 10.0
MAX_ITERS = 50
vif_cols = feat_cols_pruned.copy()

for iteration in range(MAX_ITERS):
    if len(vif_cols) <= 2:
        break
    X_vif = master_final[vif_cols].values
    # Add constant for VIF computation
    X_vif_c = np.column_stack([np.ones(len(X_vif)), X_vif])
    vifs = []
    for i in range(1, X_vif_c.shape[1]):  # skip constant
        try:
            v = variance_inflation_factor(X_vif_c, i)
        except Exception:
            v = np.inf
        vifs.append(v)
    max_vif = max(vifs)
    if max_vif <= MAX_VIF:
        break
    worst_idx = vifs.index(max_vif)
    worst_col = vif_cols[worst_idx]
    vif_cols.remove(worst_col)

print(f"After VIF removal (VIF<{MAX_VIF}): {len(vif_cols)} features")

# Final selected features
selected_features = vif_cols
print(f"\nFinal selected features ({len(selected_features)}):")
for f in selected_features:
    print(f"  - {f}")

# Build final dataset with selected features + metadata
final_cols = meta_cols + selected_features
master_final = master_final[final_cols]

# Show strongest correlations with the differenced target
numeric_corr = master_final.corr()
print(f"\nTop 10 correlated features with {active_target}:")
print(numeric_corr[active_target].abs().sort_values(ascending=False).head(10))

# Save the final dataset
output_path = os.path.join(PROCESSED_DATA_DIR, 'final_feature_set.csv')
master_final.to_csv(output_path)
print(f"\nFinal Feature Set saved to {output_path} with shape {master_final.shape}")


After lag creation and dropna: (29, 126)

--- Feature Selection ---
Starting features: 120
After correlation pruning (|r|>0.90): 72 features (48 dropped)


After VIF removal (VIF<10.0): 22 features

Final selected features (22):
  - Production_lag1
  - Annual_Rainfall_lag1
  - Annual_Rainfall_lag2
  - Annual_Rainfall_roll3_mean
  - Temperature_lag1
  - Temperature_lag2
  - Temperature_roll3_mean
  - Total_freshwater_withdrawals_billion_m3_diff_roll3_mean
  - Agriculture_share_of_withdrawals_pct_diff_lag2
  - Agri_Electricity_GWh_diff_lag1
  - Agri_Electricity_GWh_diff_lag2
  - Renewable_Energy_Use_in_Agriculture_GWh_diff_lag2
  - NonRenewable_Energy_Use_in_Agriculture_GWh_diff_lag1
  - NonRenewable_Energy_Use_in_Agriculture_GWh_diff_roll3_mean
  - Fertilizer_diff_lag1
  - Fertilizer_diff_lag2
  - cereal_yield_diff_lag1
  - cereal_yield_diff_roll3_mean
  - Agricultural_land_ha_final_diff_lag1
  - Agricultural_land_ha_final_diff_roll3_mean
  - Estimated_irrigated_area_ha_diff_lag1
  - Estimated_irrigated_area_ha_diff_roll3_mean

Top 10 correlated features with Target_Total_CO2e_diff:
Target_Total_CO2e_diff                                   

### Key Findings
Based on the feature engineering and target alignment process:

1. **Target variable** constructed as CO₂ + CH₄ + N₂O total equivalent emissions.
2. **ADF test** confirmed non-stationarity (p ≈ 0.99); 1st-order differencing applied.
3. **Interpolation fix**: First NaN row after differencing is dropped (not interpolated) to prevent data fabrication.
4. **Feature selection**: Two-stage correlation pruning (|r| > 0.90) and VIF removal (VIF < 10) reduces features to ≤ 20.
5. **Lag and rolling features** (t−1, t−2, 3-period rolling mean) capture temporal dynamics.
6. **Deterministic trend variables** (Year_Code, StartYear, Crop_Year) excluded to force learning from physical covariates.


---
### End of Notebook 02 — Feature Engineering and Target Alignment

**Outputs produced:**
- `../data/processed/final_feature_set.csv` — final engineered dataset ready for modeling


**CRITICAL NOTE FOR NOTEBOOK 03:**
The primary modeling target (`Target_Total_CO2e_diff`) has been 1st-order differenced to achieve stationarity. Original absolute levels (`Target_Total_CO2e`) are preserved in the dataset so that the models in Notebook 03 can perform an **inverse transformation**. This is required to output predictions in actual metric tonnage ($\Delta CO_2e$ + $Last\_Actual\_CO_2e$) for the "Results" section of the paper.

**Next step → Notebook 03:** Machine Learning Modeling & Impact Analysis (SHAP). Building baseline and advanced predictive models (Linear Regression, XGBoost) and analyzing feature importance.

<div align="center"><sub>END OF NOTEBOOK 02</sub></div>